# Gait Anomaly Detection / 1D Convolutional Autoencoder

By [@dhia9](https://github.com/dhia9) and me. See the [README](README.md) for context on the approach and architecture.

> **Goal:** Learn a compressed representation of *normal* walking gait from shoe-insert IMU Euler angles (yaw, pitch, roll). Anomalies are detected as gait cycles with high reconstruction error with no labels required for the initial model.

This notebook is the runnable pipeline: synthetic data to play with, gait-cycle segmentation, autoencoder training, and anomaly flagging. Run it top to bottom.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
from scipy.interpolate import interp1d
from pathlib import Path
import warnings, copy

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (14, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
})
print(f"PyTorch {torch.__version__}  |  device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1 · Configuration

All the tuneable parameters are grouped here. If you're working with a new
sensor, the first things to adjust are `MIN_STRIDE_SAMPLES` and
`PITCH_PEAK_PROMINENCE`. These control how the pipeline chops the signal into
gait cycles, and they depend on your IMU's sampling rate and how it's mounted
in the shoe.

In [ ]:
# ── Sensor and segmentation ─────────────────────────────────────
SAMPLING_RATE       = 100          # Hz (match your IMU)
CYCLE_LENGTH        = 128          # resampled timesteps per cycle
MIN_STRIDE_SAMPLES  = 50           # min samples between heel strikes
PITCH_PEAK_PROMINENCE = 8.0        # degrees, tune per sensor mount
LOWPASS_CUTOFF      = 15.0         # Hz (set to 0 to disable)

# ── Model ──────────────────────────────────────────────────────
IN_CHANNELS  = 3                   # yaw, pitch, roll
LATENT_DIM   = 32
BATCH_SIZE   = 64
EPOCHS       = 120
LR           = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE     = 15                  # early-stopping patience
TRAIN_SPLIT  = 0.8                 # fraction of cycles for training

# ── Anomaly detection ──────────────────────────────────────────
ANOMALY_PERCENTILE = 97            # threshold on normal reconstruction error

# ── Misc ───────────────────────────────────────────────────────
SEED   = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(SEED)
torch.manual_seed(SEED)

## 2 · Data loading

### 2a: Synthetic data generator *(for testing)*
This function creates a fake but realistic-looking foot-mounted IMU signal so you can run the entire pipeline before collecting real data. It was super helpful for debugging. Skip this cell once you have actual CSV files.

In [ ]:
def generate_synthetic_gait(duration_s=120, fs=100, inject_anomalies=True):
    """
    Generate a DataFrame that mimics a foot-mounted IMU recording.

    Parameters
    ----------
    duration_s : float   - total recording length in seconds
    fs         : int     - sampling rate (Hz)
    inject_anomalies : bool - if True, corrupt ~5% of strides

    Returns
    -------
    pd.DataFrame with columns: Relative timestamp, yaw, pitch, roll
    """
    n = int(duration_s * fs)
    t = np.arange(n) / fs
    stride_freq = 0.9                       # ~1.1 s per stride

    # pitch: dominant sagittal-plane signal
    pitch = (15.0 * np.sin(2 * np.pi * stride_freq * t)
           +  7.0 * np.sin(4 * np.pi * stride_freq * t - 0.4)
           +  2.5 * np.sin(6 * np.pi * stride_freq * t + 0.2))

    # roll: smaller frontal-plane oscillation
    roll  = (5.0 * np.sin(2 * np.pi * stride_freq * t + 0.5)
           + 2.0 * np.sin(4 * np.pi * stride_freq * t + 0.3))

    # yaw: slow drift + small oscillation
    yaw   = (0.005 * t
           + 3.0 * np.sin(2 * np.pi * stride_freq * t + 1.0))

    # add sensor noise
    pitch += np.random.randn(n) * 0.8
    roll  += np.random.randn(n) * 0.5
    yaw   += np.random.randn(n) * 0.4

    # inject a few anomalous strides
    if inject_anomalies:
        stride_samples = int(fs / stride_freq)
        n_strides = n // stride_samples
        anomaly_idxs = np.random.choice(n_strides, size=max(1, n_strides // 20), replace=False)
        for si in anomaly_idxs:
            s = si * stride_samples
            e = min(s + stride_samples, n)
            kind = np.random.choice(["asymmetry", "foot_drop", "stumble"])
            if kind == "asymmetry":
                roll[s:e] += 12 * np.sin(2 * np.pi * stride_freq * t[s:e])
            elif kind == "foot_drop":
                pitch[s:e] *= 0.35
            else:
                pitch[s:e] += np.random.randn(e - s) * 8

    return pd.DataFrame({
        "Relative timestamp": t,
        "yaw": yaw,
        "pitch": pitch,
        "roll": roll,
    })

# Generate a 2-minute synthetic recording to test the pipeline
df = generate_synthetic_gait(duration_s=120, fs=SAMPLING_RATE)
print(f"Synthetic data: {len(df)} samples, {len(df)/SAMPLING_RATE:.1f} s")
df.head()

### 2b: Load your real CSV

Uncomment the cell below and point it at your file. The expected format is four
columns: `Relative timestamp, yaw, pitch, roll` (floats, angles in degrees).

In [ ]:
# Uncomment to load your own data
# DATA_PATH = Path("your_recording.csv")
# df = pd.read_csv(DATA_PATH)
#
# # Infer sampling rate from timestamps
# dt = df["Relative timestamp"].diff().median()
# SAMPLING_RATE = int(round(1.0 / dt))
# print(f"Loaded {len(df)} samples at ~{SAMPLING_RATE} Hz "
#       f"({len(df)/SAMPLING_RATE:.1f} s)")
# df.head()

## 3 · Visualise raw signals

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
t = df["Relative timestamp"].values

for ax, col, color in zip(axes, ["yaw", "pitch", "roll"],
                           ["#4C72B0", "#DD8452", "#55A868"]):
    ax.plot(t, df[col].values, color=color, lw=0.6, alpha=0.9)
    ax.set_ylabel(f"{col} (deg)")
    ax.set_title(col.capitalize(), fontsize=11, fontweight="bold")

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Raw Euler angles from shoe-insert IMU", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

# Zoom into a few strides to see the waveform shape
fig, axes = plt.subplots(3, 1, figsize=(14, 5), sharex=True)
mask = (t >= 5) & (t <= 12)
for ax, col, color in zip(axes, ["yaw", "pitch", "roll"],
                           ["#4C72B0", "#DD8452", "#55A868"]):
    ax.plot(t[mask], df[col].values[mask], color=color, lw=1.2)
    ax.set_ylabel(f"{col} (deg)")
axes[-1].set_xlabel("Time (s)")
fig.suptitle("Zoomed in: ~6 strides", fontsize=12)
fig.tight_layout()
plt.show()

## 4 · Low-pass filter (optional)

Set `LOWPASS_CUTOFF = 0` to skip.

In [ ]:
def lowpass(signal, cutoff, fs, order=4):
    if cutoff <= 0 or cutoff >= fs / 2:
        return signal
    b, a = butter(order, cutoff / (fs / 2), btype="low")
    return filtfilt(b, a, signal)

angles = np.column_stack([df["yaw"].values,
                          df["pitch"].values,
                          df["roll"].values])      # (N, 3)
timestamps = df["Relative timestamp"].values

if LOWPASS_CUTOFF > 0:
    for ch in range(3):
        angles[:, ch] = lowpass(angles[:, ch], LOWPASS_CUTOFF, SAMPLING_RATE)
    print(f"Applied {LOWPASS_CUTOFF} Hz low-pass filter")
else:
    print("Low-pass filter disabled")

## 5 · Gait-cycle segmentation

This is where we detect **heel strikes** as prominent peaks in the pitch
channel. The pitch angle (sagittal plane) has the strongest periodic signal
during walking, so it's the best channel to segment on. Each pair of
consecutive heel strikes defines one gait cycle.

> **Tip :** If the segmentation looks off, tweak `PITCH_PEAK_PROMINENCE` and `MIN_STRIDE_SAMPLES` and re-check the plot below.

In [ ]:
pitch = angles[:, 1]                           # column index 1 = pitch

# Detect heel strikes as local maxima in pitch
heel_strikes, properties = find_peaks(
    pitch,
    distance=MIN_STRIDE_SAMPLES,
    prominence=PITCH_PEAK_PROMINENCE,
)

print(f"Detected {len(heel_strikes)} heel strikes "
      f"-> {len(heel_strikes) - 1} gait cycles")

# Quick sanity check on stride durations
stride_durations = np.diff(timestamps[heel_strikes])
print(f"Stride duration: "
      f"mean={stride_durations.mean():.3f} s, "
      f"std={stride_durations.std():.3f} s, "
      f"range=[{stride_durations.min():.3f}, {stride_durations.max():.3f}] s")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
t_sec = timestamps

# Show a zoomed segment so individual peaks are visible
view_start, view_end = 3.0, 14.0
mask = (t_sec >= view_start) & (t_sec <= view_end)

ax.plot(t_sec[mask], pitch[mask], lw=1.2, label="Pitch")

hs_mask = (t_sec[heel_strikes] >= view_start) & (t_sec[heel_strikes] <= view_end)
ax.plot(t_sec[heel_strikes[hs_mask]], pitch[heel_strikes[hs_mask]],
        "rv", ms=10, label="Heel strike")

ax.set_xlabel("Time (s)")
ax.set_ylabel("Pitch (deg)")
ax.set_title("Heel-strike detection on pitch signal")
ax.legend()
fig.tight_layout()
plt.show()

## 6 · Extract and resample gait cycles

Each cycle gets resampled to `CYCLE_LENGTH` timesteps using linear
interpolation. This way the autoencoder always sees fixed-size inputs
regardless of how fast the person was walking.

In [ ]:
def extract_cycles(angles, heel_strikes, target_len):
    """
    Cut the angle array at heel-strike indices and resample
    each cycle to target_len timesteps.

    Returns
    -------
    cycles  : np.ndarray, shape (n_cycles, target_len, 3)
    lengths : list[int] - original sample count per cycle
    """
    cycles, lengths = [], []
    for i in range(len(heel_strikes) - 1):
        start = heel_strikes[i]
        end   = heel_strikes[i + 1]
        seg   = angles[start:end]               # (variable_len, 3)
        n     = len(seg)
        if n < 20:                               # skip tiny fragments
            continue
        x_old = np.linspace(0, 1, n)
        x_new = np.linspace(0, 1, target_len)
        resampled = np.zeros((target_len, 3))
        for ch in range(3):
            resampled[:, ch] = interp1d(x_old, seg[:, ch], kind="linear")(x_new)
        cycles.append(resampled)
        lengths.append(n)
    return np.array(cycles, dtype=np.float32), lengths

cycles, orig_lengths = extract_cycles(angles, heel_strikes, CYCLE_LENGTH)
print(f"Extracted {len(cycles)} gait cycles, each resampled to "
      f"{CYCLE_LENGTH} timesteps x {cycles.shape[2]} channels")

### Channel-wise z-score normalisation

We compute mean and std **per channel** across the training set and apply the
same transform at inference. Make sure to save these stats, because you'll need
them later when running the model on new data.

In [ ]:
# Compute stats across (n_cycles, timesteps) per channel
chan_mean = cycles.mean(axis=(0, 1))            # shape (3,)
chan_std  = cycles.std(axis=(0, 1))             # shape (3,)
chan_std[chan_std < 1e-8] = 1.0                 # guard against zero-variance

cycles_norm = (cycles - chan_mean) / chan_std

print("Channel statistics (before normalisation):")
for i, name in enumerate(["yaw", "pitch", "roll"]):
    print(f"  {name:>5s}  mean={chan_mean[i]:+.2f} deg  std={chan_std[i]:.2f} deg")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.linspace(0, 100, CYCLE_LENGTH)          # % of gait cycle

for ch, (ax, name) in enumerate(zip(axes, ["Yaw", "Pitch", "Roll"])):
    for c in cycles_norm[::3]:                  # plot every 3rd for clarity
        ax.plot(x, c[:, ch], alpha=0.15, lw=0.8, color="#4C72B0")
    ax.plot(x, cycles_norm[:, :, ch].mean(axis=0),
            color="#C44E52", lw=2, label="Mean")
    ax.set_title(name)
    ax.set_xlabel("Gait cycle (%)")
    ax.set_ylabel("Normalised angle")
    ax.legend(fontsize=9)

fig.suptitle("All extracted gait cycles overlaid (normalised)", fontsize=12)
fig.tight_layout()
plt.show()

## 7 · PyTorch dataset and data loaders

In [ ]:
class GaitCycleDataset(Dataset):
    """Wraps a numpy array of gait cycles into a PyTorch Dataset."""

    def __init__(self, cycles: np.ndarray):
        # Store as (N, channels, timesteps) for Conv1d
        self.data = torch.from_numpy(cycles).permute(0, 2, 1).float()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]                    # (3, 128)


full_dataset = GaitCycleDataset(cycles_norm)

# Train / validation split
n_train = int(len(full_dataset) * TRAIN_SPLIT)
n_val   = len(full_dataset) - n_train
train_ds, val_ds = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"Train: {n_train}  |  Val: {n_val}  |  Batch size: {BATCH_SIZE}")

## 8 · 1D Convolutional Autoencoder

See the [README](README.md#model-architecture) for the full architecture
diagram. Each Conv block is Conv1d + BatchNorm + LeakyReLU, stride 2.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, stride, padding):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=padding),
            nn.BatchNorm1d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, stride, padding, out_padding):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(in_ch, out_ch, kernel,
                               stride=stride, padding=padding,
                               output_padding=out_padding),
            nn.BatchNorm1d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class GaitAutoencoder(nn.Module):
    """
    1D convolutional autoencoder for gait cycles.

    Input / output shape : (batch, 3, 128)
    Latent shape         : (batch, LATENT_DIM)
    """

    def __init__(self, in_channels=3, latent_dim=32):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            ConvBlock(in_channels, 32,  7, stride=2, padding=3),   # -> (32,  64)
            ConvBlock(32,          64,  5, stride=2, padding=2),   # -> (64,  32)
            ConvBlock(64,          128, 3, stride=2, padding=1),   # -> (128, 16)
            ConvBlock(128,         256, 3, stride=2, padding=1),   # -> (256,  8)
        )
        self.enc_flat_dim = 256 * 8
        self.fc_enc = nn.Linear(self.enc_flat_dim, latent_dim)

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, self.enc_flat_dim)
        self.decoder = nn.Sequential(
            DeconvBlock(256, 128, 3, stride=2, padding=1, out_padding=1),  # -> (128, 16)
            DeconvBlock(128, 64,  3, stride=2, padding=1, out_padding=1),  # -> (64,  32)
            DeconvBlock(64,  32,  5, stride=2, padding=2, out_padding=1),  # -> (32,  64)
            nn.ConvTranspose1d(32, in_channels, 7,
                               stride=2, padding=3, output_padding=1),     # -> (3,  128)
        )

    def encode(self, x):
        h = self.encoder(x)                      # (B, 256, 8)
        h = h.view(h.size(0), -1)                # (B, 2048)
        return self.fc_enc(h)                     # (B, latent_dim)

    def decode(self, z):
        h = self.fc_dec(z)                        # (B, 2048)
        h = h.view(h.size(0), 256, 8)             # (B, 256, 8)
        return self.decoder(h)                    # (B, 3, 128)

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z


model = GaitAutoencoder(in_channels=IN_CHANNELS, latent_dim=LATENT_DIM).to(DEVICE)

# Quick shape check
dummy = torch.randn(2, IN_CHANNELS, CYCLE_LENGTH, device=DEVICE)
out, z = model(dummy)
print(f"Input:   {tuple(dummy.shape)}")
print(f"Latent:  {tuple(z.shape)}")
print(f"Output:  {tuple(out.shape)}")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

## 9 · Training loop

Standard MSE reconstruction loss with early stopping on the validation set.
The scheduler halves the learning rate when the validation loss plateaus.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=7, verbose=False,
)
criterion = nn.MSELoss()

history = {"train_loss": [], "val_loss": []}
best_val  = float("inf")
best_state = None
wait = 0

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        x_hat, _ = model(batch)
        loss = criterion(x_hat, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    train_loss = epoch_loss / n_train

    # Validate
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            x_hat, _ = model(batch)
            val_loss_sum += criterion(x_hat, batch).item() * batch.size(0)
    val_loss = val_loss_sum / n_val

    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Early stopping
    if val_loss < best_val:
        best_val   = val_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1

    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch:>3d}/{EPOCHS}  "
              f"train={train_loss:.6f}  val={val_loss:.6f}  "
              f"lr={lr_now:.1e}  patience={wait}/{PATIENCE}")

    if wait >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# Restore best weights
model.load_state_dict(best_state)
print(f"\nBest validation loss: {best_val:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["train_loss"], label="Train", lw=1.5)
ax.plot(history["val_loss"],   label="Validation", lw=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training curves")
ax.legend()
fig.tight_layout()
plt.show()

## 10 · Anomaly detection

Now for the fun part. We compute the reconstruction error per cycle on the
full dataset and look at the distribution. Most cycles should cluster at low
error (these are the normal ones). We set the anomaly threshold at the chosen
percentile, and anything above it gets flagged.

In [ ]:
def compute_reconstruction_errors(model, dataset, device):
    """Return per-sample MSE reconstruction error."""
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    errors = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            x_hat, _ = model(batch)
            mse = ((x_hat - batch) ** 2).mean(dim=(1, 2))   # per sample
            errors.append(mse.cpu().numpy())
    return np.concatenate(errors)

errors = compute_reconstruction_errors(model, full_dataset, DEVICE)

threshold = np.percentile(errors, ANOMALY_PERCENTILE)
n_anomalies = (errors > threshold).sum()

print(f"Reconstruction error |  "
      f"mean: {errors.mean():.5f},  std: {errors.std():.5f}")
print(f"Threshold ({ANOMALY_PERCENTILE}th percentile): {threshold:.5f}")
print(f"Flagged anomalies: {n_anomalies} / {len(errors)} "
      f"({100*n_anomalies/len(errors):.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(errors, bins=60, alpha=0.75, color="#4C72B0", edgecolor="white")
ax.axvline(threshold, color="#C44E52", ls="--", lw=2,
           label=f"Threshold ({ANOMALY_PERCENTILE}th pctl) = {threshold:.4f}")
ax.set_xlabel("Reconstruction error (MSE)")
ax.set_ylabel("Count")
ax.set_title("Per-cycle reconstruction error distribution")
ax.legend(fontsize=10)
fig.tight_layout()
plt.show()

### Normal vs. flagged cycles side by side

In [ ]:
normal_idxs  = np.where(errors <= threshold)[0]
anomaly_idxs = np.where(errors >  threshold)[0]

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey="col")
channel_names = ["Yaw", "Pitch", "Roll"]
x_pct = np.linspace(0, 100, CYCLE_LENGTH)

# Top row: a few normal cycles
for ch in range(3):
    ax = axes[0, ch]
    for idx in normal_idxs[:8]:
        cycle = full_dataset[idx].numpy()          # (3, 128)
        ax.plot(x_pct, cycle[ch], alpha=0.5, lw=0.9)
    ax.set_title(f"{channel_names[ch]} (Normal)", fontsize=11)
    if ch == 0:
        ax.set_ylabel("Normalised angle")

# Bottom row: flagged anomalies
for ch in range(3):
    ax = axes[1, ch]
    for idx in anomaly_idxs[:8]:
        cycle = full_dataset[idx].numpy()
        ax.plot(x_pct, cycle[ch], alpha=0.5, lw=0.9)
    ax.set_title(f"{channel_names[ch]} (Anomaly)", fontsize=11)
    ax.set_xlabel("Gait cycle (%)")
    if ch == 0:
        ax.set_ylabel("Normalised angle")

fig.suptitle("Normal gait cycles vs. flagged anomalies", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

### Reconstruction quality: input vs. output

This is a good way to build intuition for what the autoencoder is actually
doing. For normal cycles, the dashed line (reconstruction) should track the
solid line (input) closely. For anomalies, you'll see the reconstruction
diverge because the model hasn't learned that pattern.

In [ ]:
def plot_reconstruction(model, dataset, idx, device):
    model.eval()
    x = dataset[idx].unsqueeze(0).to(device)
    with torch.no_grad():
        x_hat, z = model(x)
    x     = x.cpu().squeeze().numpy()              # (3, 128)
    x_hat = x_hat.cpu().squeeze().numpy()
    err   = ((x - x_hat) ** 2).mean()

    fig, axes = plt.subplots(1, 3, figsize=(15, 3))
    for ch, (ax, name) in enumerate(zip(axes, ["Yaw", "Pitch", "Roll"])):
        ax.plot(x[ch],     lw=1.5, label="Input")
        ax.plot(x_hat[ch], lw=1.5, label="Reconstruction", ls="--")
        ax.set_title(name)
        ax.legend(fontsize=8)
    fig.suptitle(f"Cycle #{idx}  |  MSE = {err:.5f}", fontsize=11)
    fig.tight_layout()
    plt.show()

# Show one normal cycle and one anomalous one
print("== Normal cycle ==")
plot_reconstruction(model, full_dataset, normal_idxs[0], DEVICE)

if len(anomaly_idxs) > 0:
    print("\n== Anomalous cycle ==")
    plot_reconstruction(model, full_dataset, anomaly_idxs[0], DEVICE)

## 11 · Latent-space visualisation (t-SNE)

This was one of my favourite parts. Projecting the 32-d latent vectors down to
2D with t-SNE shows whether the normal and anomalous cycles actually land in
different regions. You can also see natural sub-clusters forming in the data,
probably from different walking speeds or left vs. right foot. It really makes
the model feel less like a black box.

In [ ]:
from sklearn.manifold import TSNE

# Encode all cycles
loader = DataLoader(full_dataset, batch_size=256, shuffle=False)
latents = []
model.eval()
with torch.no_grad():
    for batch in loader:
        z = model.encode(batch.to(DEVICE))
        latents.append(z.cpu().numpy())
latents = np.concatenate(latents)

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
proj = tsne.fit_transform(latents)

labels = np.array(["Normal"] * len(errors))
labels[anomaly_idxs] = "Anomaly"

fig, ax = plt.subplots(figsize=(8, 6))
for label, color, marker in [("Normal", "#4C72B0", "o"), ("Anomaly", "#C44E52", "X")]:
    mask = labels == label
    ax.scatter(proj[mask, 0], proj[mask, 1], c=color, marker=marker,
               s=20 if label == "Normal" else 60, alpha=0.6, label=label, edgecolors="white", lw=0.3)
ax.set_title("t-SNE of autoencoder latent space")
ax.legend()
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
fig.tight_layout()
plt.show()

## 12 · Inference on new recordings

This function wraps the full pipeline into a single call: load the CSV, filter,
segment, resample, normalise, reconstruct, and flag. It returns a DataFrame with
one row per detected gait cycle so you can quickly see which ones were flagged.

In [ ]:
def analyse_recording(csv_path, model, chan_mean, chan_std, threshold,
                      fs=SAMPLING_RATE, device=DEVICE):
    """
    End-to-end inference on a new CSV recording.

    Returns a DataFrame with columns:
        cycle_index, start_time, end_time, duration_s,
        reconstruction_error, is_anomaly
    """
    df_new = pd.read_csv(csv_path)
    t  = df_new["Relative timestamp"].values
    ang = np.column_stack([df_new["yaw"].values,
                           df_new["pitch"].values,
                           df_new["roll"].values])

    # Filter
    if LOWPASS_CUTOFF > 0:
        for ch in range(3):
            ang[:, ch] = lowpass(ang[:, ch], LOWPASS_CUTOFF, fs)

    # Segment
    peaks, _ = find_peaks(ang[:, 1], distance=MIN_STRIDE_SAMPLES,
                          prominence=PITCH_PEAK_PROMINENCE)
    cycles, _ = extract_cycles(ang, peaks, CYCLE_LENGTH)

    # Normalise with training stats
    cycles_n = (cycles - chan_mean) / chan_std

    # Reconstruct
    ds = GaitCycleDataset(cycles_n)
    errs = compute_reconstruction_errors(model, ds, device)

    # Build results table
    rows = []
    for i in range(len(cycles)):
        rows.append({
            "cycle_index": i,
            "start_time":  t[peaks[i]],
            "end_time":    t[peaks[i + 1]],
            "duration_s":  t[peaks[i + 1]] - t[peaks[i]],
            "reconstruction_error": errs[i],
            "is_anomaly":  errs[i] > threshold,
        })
    return pd.DataFrame(rows)

# Example usage (uncomment with a real file)
# results = analyse_recording("new_walk.csv", model, chan_mean, chan_std, threshold)
# print(results[results["is_anomaly"]])

## 13 · Per-channel error breakdown

When a cycle gets flagged, it's useful to know *which* channel is responsible
for the high error. This can hint at what kind of gait deviation is happening.
For example, a pitch anomaly points to a sagittal-plane issue, while a roll
anomaly suggests lateral instability.

In [ ]:
def per_channel_errors(model, dataset, device):
    """Return (N, 3) array of per-channel MSE for each cycle."""
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    ch_errors = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            x_hat, _ = model(batch)
            mse = ((x_hat - batch) ** 2).mean(dim=2)     # (B, 3)
            ch_errors.append(mse.cpu().numpy())
    return np.concatenate(ch_errors)

ch_errs = per_channel_errors(model, full_dataset, DEVICE)

fig, ax = plt.subplots(figsize=(10, 4))
positions = np.arange(len(ch_errs))
bottom = np.zeros(len(ch_errs))
colors = ["#4C72B0", "#DD8452", "#55A868"]

# Sort by total error for a cleaner plot
order = np.argsort(errors)
for ch, name, color in zip(range(3), ["Yaw", "Pitch", "Roll"], colors):
    vals = ch_errs[order, ch]
    ax.bar(positions, vals, bottom=bottom, color=color, width=1.0,
           alpha=0.85, label=name)
    bottom += vals

ax.axhline(threshold, color="#C44E52", ls="--", lw=1.5, label="Anomaly threshold")
ax.set_xlabel("Cycle (sorted by total error)")
ax.set_ylabel("Reconstruction error")
ax.set_title("Per-channel error breakdown across all cycles")
ax.legend()
fig.tight_layout()
plt.show()

## 14 · Save / load the trained model

In [ ]:
SAVE_DIR = Path("checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "latent_dim":       LATENT_DIM,
    "in_channels":      IN_CHANNELS,
    "cycle_length":     CYCLE_LENGTH,
    "chan_mean":         chan_mean,
    "chan_std":          chan_std,
    "threshold":        threshold,
    "anomaly_percentile": ANOMALY_PERCENTILE,
    "config": {
        "sampling_rate":        SAMPLING_RATE,
        "min_stride_samples":   MIN_STRIDE_SAMPLES,
        "pitch_peak_prominence": PITCH_PEAK_PROMINENCE,
        "lowpass_cutoff":       LOWPASS_CUTOFF,
    },
}
torch.save(checkpoint, SAVE_DIR / "gait_autoencoder.pt")
print(f"Checkpoint saved to {SAVE_DIR / 'gait_autoencoder.pt'}")

In [ ]:
# Load a saved checkpoint
def load_model(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = GaitAutoencoder(
        in_channels=ckpt["in_channels"],
        latent_dim=ckpt["latent_dim"],
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, ckpt

# model, ckpt = load_model("checkpoints/gait_autoencoder.pt", DEVICE)
# threshold = ckpt["threshold"]
# chan_mean  = ckpt["chan_mean"]
# chan_std   = ckpt["chan_std"]

## 15 · Next steps

See the [README roadmap](README.md#roadmap--ideas-for-future-work) for the
full list. The highest-impact next moves are collecting more diverse normal data
and starting the active labelling loop (have someone review just the flagged
cycles). After a few hundred labels per anomaly class, a simple classifier on
the latent embeddings should work well.
